In [ ]:
!pip install --no-index --find-links /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels arc-agi python-dotenv


In [ ]:
%%writefile /kaggle/working/my_agent.py
"""Auto-generated by arc_prize_2026.arc3.kaggle.build_notebook — do not edit; edit the sources in src/."""
from __future__ import annotations
import random
from collections import Counter, defaultdict, deque
from dataclasses import dataclass, field
from typing import Any
import numpy as np
from arcengine import FrameData, GameAction, GameState


# ---- perception.py ----
"""Frame -> objects -> abstract state key.

An ARC-AGI-3 frame is a 64x64 grid of colours 0..15. We segment it into same-colour 4-connected
components. The background is the largest component (by area). Objects carry colour, bbox, area and a
shape hash (mask relative to its bbox), so a moved object keeps its identity and a recoloured object
keeps its shape. The abstract state key is the sorted tuple of (colour, y0, x0, shape_hash) for all
non-background objects; two frames with the same key are the same abstract state.
"""



H = W = 64


@dataclass(frozen=True)
class Obj:
    colour: int
    y0: int
    x0: int
    y1: int  # exclusive
    x1: int
    area: int
    shape: int  # hash of the boolean mask inside the bbox
    mask: bytes = field(default=b"", compare=False, hash=False, repr=False)  # packed bool mask, for rendering

    @property
    def h(self) -> int:
        return self.y1 - self.y0

    @property
    def w(self) -> int:
        return self.x1 - self.x0

    @property
    def centre(self) -> tuple[int, int]:
        return ((self.y0 + self.y1 - 1) // 2, (self.x0 + self.x1 - 1) // 2)


def label_components(grid: np.ndarray) -> tuple[np.ndarray, int]:
    """4-connected same-colour labelling. Returns (labels, n) with labels in 1..n."""
    h, w = grid.shape
    labels = np.zeros((h, w), dtype=np.int32)
    n = 0
    for y in range(h):
        row = grid[y]
        for x in range(w):
            if labels[y, x]:
                continue
            n += 1
            c = row[x]
            stack = [(y, x)]
            labels[y, x] = n
            while stack:
                cy, cx = stack.pop()
                if cy > 0 and not labels[cy - 1, cx] and grid[cy - 1, cx] == c:
                    labels[cy - 1, cx] = n; stack.append((cy - 1, cx))
                if cy < h - 1 and not labels[cy + 1, cx] and grid[cy + 1, cx] == c:
                    labels[cy + 1, cx] = n; stack.append((cy + 1, cx))
                if cx > 0 and not labels[cy, cx - 1] and grid[cy, cx - 1] == c:
                    labels[cy, cx - 1] = n; stack.append((cy, cx - 1))
                if cx < w - 1 and not labels[cy, cx + 1] and grid[cy, cx + 1] == c:
                    labels[cy, cx + 1] = n; stack.append((cy, cx + 1))
    return labels, n


def segment(grid: np.ndarray, drop_background: bool = True) -> list[Obj]:
    grid = np.asarray(grid, dtype=np.int16)
    labels, n = label_components(grid)
    objs = []
    flat = labels.ravel()
    order = np.argsort(flat, kind="stable")
    sorted_labels = flat[order]
    bounds = np.searchsorted(sorted_labels, np.arange(1, n + 2))
    ys_all, xs_all = np.divmod(order, grid.shape[1])
    for k in range(n):
        idx = slice(bounds[k], bounds[k + 1])
        ys, xs = ys_all[idx], xs_all[idx]
        y0, y1, x0, x1 = int(ys.min()), int(ys.max()) + 1, int(xs.min()), int(xs.max()) + 1
        mask = np.zeros((y1 - y0, x1 - x0), dtype=bool)
        mask[ys - y0, xs - x0] = True
        objs.append(Obj(int(grid[ys[0], xs[0]]), y0, x0, y1, x1, int(len(ys)), hash(mask.tobytes()) ^ (mask.shape[0] << 20) ^ mask.shape[1], mask.tobytes()))
    if drop_background and objs:
        bg = max(objs, key=lambda o: o.area)
        objs = [o for o in objs if o is not bg]
    objs.sort(key=lambda o: (o.y0, o.x0, o.colour, o.shape))
    return objs


def ident(o: Obj) -> tuple[int, int, int]:
    """Object class used for learning effects: colour and bbox size (robust to small occlusions)."""
    return (o.colour, o.h, o.w)


def state_key(objs: list[Obj]) -> int:
    """Identity by colour + bbox; a hole carved into a floor by a sprite does not change the floor."""
    return hash(tuple((o.colour, o.y0, o.x0, o.y1, o.x1) for o in objs))


def frame_key(grid) -> int:
    return hash(np.asarray(grid, dtype=np.int8).tobytes())


def diff(before: list[Obj], after: list[Obj], max_disp: int = 32) -> dict:
    """Object-level diff. same = equal colour+bbox; moved = same class (colour,h,w), nearest within max_disp;
    recoloured = equal bbox, different colour; the rest appeared/vanished."""
    b_left, a_left = list(before), list(after)
    same, moved_, recol = [], [], []
    a_index = {(p.colour, p.y0, p.x0, p.y1, p.x1): p for p in a_left}
    for o in list(b_left):
        p = a_index.get((o.colour, o.y0, o.x0, o.y1, o.x1))
        if p is not None and p in a_left:
            same.append(o); b_left.remove(o); a_left.remove(p)
    for o in list(b_left):
        best = None
        for p in a_left:
            if ident(o) == ident(p):
                d = abs(p.y0 - o.y0) + abs(p.x0 - o.x0)
                if d <= max_disp and (best is None or d < best[0]):
                    best = (d, p)
        if best is not None:
            p = best[1]; moved_.append((o, p, (p.y0 - o.y0, p.x0 - o.x0))); b_left.remove(o); a_left.remove(p)
    for o in list(b_left):
        for p in list(a_left):
            if (o.y0, o.x0, o.y1, o.x1) == (p.y0, p.x0, p.y1, p.x1):
                recol.append((o, p)); b_left.remove(o); a_left.remove(p); break
    return {"same": same, "moved": moved_, "recoloured": recol, "vanished": b_left, "appeared": a_left}

def moved(o: Obj, dy: int, dx: int) -> Obj:
    return Obj(o.colour, o.y0 + dy, o.x0 + dx, o.y1 + dy, o.x1 + dx, o.area, o.shape, o.mask)


def render_move(grid: np.ndarray, o: Obj, new: Obj, bg: int) -> np.ndarray:
    """Return a copy of grid with object o erased (filled with bg) and redrawn at new's position."""
    g = grid.copy()
    m = np.frombuffer(o.mask, dtype=bool).reshape(o.h, o.w) if o.mask else np.ones((o.h, o.w), dtype=bool)
    sub = g[o.y0:o.y1, o.x0:o.x1]; sub[m] = bg
    H, W = g.shape
    if 0 <= new.y0 and new.y1 <= H and 0 <= new.x0 and new.x1 <= W:
        sub2 = g[new.y0:new.y1, new.x0:new.x1]; sub2[m] = o.colour
    return g


def background_colour(grid: np.ndarray) -> int:
    vals, counts = np.unique(grid, return_counts=True)
    return int(vals[counts.argmax()])

# ---- effects.py ----
"""Online object-level action-effect model.

Observations are (objects_before, action_key, objects_after). We learn, per action key:
  * `moves[key]`      : Counter of (colour, shape) -> Counter of displacement (dy, dx) — which object class
                        moves and by how much when this action is taken (per-object deltas from `diff`).
  * `blocked[key]`    : Counter of colours seen immediately ahead (in the learned move direction) of the
                        moving object on steps where it did NOT move (no-op for that object).
  * `passable[key]`   : Counter of colours ahead on steps where it DID move.
  * `transitions`     : exact memo (state_key, action_key) -> next state_key (deterministic games).
  * `effects[key]`    : Counter of coarse effect signatures (moved/recoloured/appeared/vanished counts),
                        used to decide whether an action does anything at all in general.
`predict(objs, key)` returns the predicted object list (or None if unknown), by applying the modal
displacement to the modal moving class when the cells ahead are background-or-passable; it returns the
same objects (no-op) when the cells ahead are in the blocked set. Only movement effects are predicted
generatively; recolour/appear/vanish are memorised through `transitions` and otherwise count as unknown.
"""




Key = object  # action key: int action id, or ("click", y//4, x//4)


def _ahead_cells(o: Obj, dy: int, dx: int, grid: np.ndarray, members: list[Obj] | None = None) -> list[int]:
    """Colours of every cell the object sweeps through when displaced by (dy, dx) in unit steps,
    excluding cells currently occupied by any member of its rigid group. [-1] marks the world edge."""
    H, W = grid.shape
    occupied = members or [o]
    steps = max(abs(dy), abs(dx))
    if steps == 0:
        return []
    sy, sx = (dy > 0) - (dy < 0), (dx > 0) - (dx < 0)
    cells: list[int] = []
    for k in range(1, steps + 1):
        ty0, tx0 = o.y0 + sy * k, o.x0 + sx * k
        ty1, tx1 = ty0 + o.h, tx0 + o.w
        if ty0 < 0 or tx0 < 0 or ty1 > H or tx1 > W:
            return [-1]
        for y in range(ty0, ty1):
            for x in range(tx0, tx1):
                if any(m.y0 <= y < m.y1 and m.x0 <= x < m.x1 for m in occupied):
                    continue
                cells.append(int(grid[y, x]))
    return cells


class EffectModel:
    def __init__(self) -> None:
        self.moves: dict[Key, Counter] = defaultdict(Counter)          # key -> Counter[((colour, shape), (dy, dx))]
        self.blocked: dict[Key, Counter] = defaultdict(Counter)
        self.passable: dict[Key, Counter] = defaultdict(Counter)
        self.transitions: dict[tuple[int, Key], int] = {}
        self.effects: dict[Key, Counter] = defaultdict(Counter)
        self.n_obs: Counter = Counter()
        self.n_pred = 0
        self.n_pred_correct = 0
        self.key_pred: dict[Key, list[int]] = defaultdict(lambda: [0, 0])  # key -> [n, correct] for generative predictions
        self.min_reliability = 0.5

    # ---- learning -------------------------------------------------------------------------------
    def observe(self, before: list[Obj], grid_before: np.ndarray, key: Key, after: list[Obj]) -> dict:
        d = diff(before, after)
        sk_b, sk_a = state_key(before), state_key(after)
        self.transitions[(sk_b, key)] = sk_a
        self.n_obs[key] += 1
        sig = (len(d["moved"]), len(d["recoloured"]), len(d["appeared"]), len(d["vanished"]))
        self.effects[key][sig] += 1
        for o, p, delta in d["moved"]:
            self.moves[key][(ident(o), delta)] += 1
            self.passable[key].update(_ahead_cells(o, *delta, grid_before))
        if not d["moved"] and sig == (0, 0, 0, 0):
            # nothing moved: if we know what this key usually moves, record what was ahead of it
            group = self.modal_group(key)
            if group is not None:
                idents, (dy, dx) = group
                members = [o for o in before if ident(o) in idents]
                for o in members:
                    self.blocked[key].update(_ahead_cells(o, dy, dx, grid_before, members))
        return d

    def modal_group(self, key: Key):
        """(set of idents that move together, modal displacement) or None."""
        if not self.moves[key]:
            return None
        by_delta = Counter()
        for (cls, delta), n in self.moves[key].items():
            by_delta[delta] += n
        delta, _ = by_delta.most_common(1)[0]
        counts = {cls: n for (cls, d), n in self.moves[key].items() if d == delta}
        top = max(counts.values())
        idents = frozenset(cls for cls, n in counts.items() if n >= 0.5 * top)
        return idents, delta

    def is_known_noop(self, key: Key) -> bool:
        c = self.effects[key]
        return sum(c.values()) >= 3 and c[(0, 0, 0, 0)] / sum(c.values()) > 0.9

    # ---- prediction -----------------------------------------------------------------------------
    def predict(self, objs: list[Obj], grid: np.ndarray, key: Key) -> tuple[list[Obj] | None, str]:
        """Return (predicted objects or None, source in {memo, model, unknown})."""
        sk = state_key(objs)
        if (sk, key) in self.transitions:
            return None, "memo"
        group = self.modal_group(key)
        if group is None:
            return None, "unknown"
        n, c = self.key_pred[key]
        if n >= 10 and c / n < self.min_reliability:
            return None, "unreliable"  # this key's mechanic is not a rigid move; memo-only from here on
        idents, (dy, dx) = group
        members = [o for o in objs if ident(o) in idents]
        if not members:
            return None, "unknown"
        bl, pa = self.blocked[key], self.passable[key]
        blocked = False
        for o in members:
            for c in _ahead_cells(o, dy, dx, grid, members):
                if c == -1 or (bl[c] > 0 and bl[c] >= pa[c]):
                    blocked = True; break
            if blocked:
                break
        if blocked:
            return list(objs), "model"
        out = [_moved(o, dy, dx) if ident(o) in idents else o for o in objs]
        return out, "model"

    def next_key(self, sk: int, key: Key) -> int | None:
        return self.transitions.get((sk, key))

    def record_prediction(self, predicted_key: int | None, observed_key: int, key: Key | None = None) -> None:
        if predicted_key is None:
            return
        self.n_pred += 1
        ok = int(predicted_key == observed_key)
        self.n_pred_correct += ok
        if key is not None:
            self.key_pred[key][0] += 1; self.key_pred[key][1] += ok

    @property
    def accuracy(self) -> float | None:
        return self.n_pred_correct / self.n_pred if self.n_pred else None

# ---- planner.py ----
"""Breadth-first planning in the learned effect model.

Nodes carry (state_key, objs, grid). Edges: for every candidate action key, the model predicts the next
objects (memo edges are followed by key only, which is enough to recognise known states but not to
expand them further unless we also saw their objects; we keep an objs cache per state key for that).
Target: the nearest state holding an untried (state, key) pair whose key class is not a known no-op.
Returns the action-key path to that state plus the exploratory key to take there.
"""





def plan_to_information(model: EffectModel, objs: list[Obj], grid: np.ndarray, keys_for, tried: dict[int, set],
                        objs_cache: dict[int, tuple[list[Obj], np.ndarray]], max_depth: int = 12, max_nodes: int = 1500):
    """keys_for(objs) -> list of (key, class_key, action, data). Returns (path_keys, final_key) or (None, None)."""
    bg = background_colour(grid)
    start = state_key(objs)
    objs_cache.setdefault(start, (objs, grid))
    q = deque([(start, [])])
    seen = {start}
    nodes = 0
    best = None
    while q and nodes < max_nodes:
        sk, path = q.popleft()
        nodes += 1
        if sk not in objs_cache:
            continue
        o_list, g = objs_cache[sk]
        cands = keys_for(o_list)
        # exploratory keys available here?
        untried = [c for c in cands if c[0] not in tried.get(sk, set()) and not model.is_known_noop(c[1])]
        if untried and path:  # don't count the start node unless nothing else exists
            untried.sort(key=lambda c: (model.n_obs[c[1]], ))
            return path, untried[0]
        if untried and not path and best is None:
            untried.sort(key=lambda c: (model.n_obs[c[1]], ))
            best = ([], untried[0])
        if len(path) >= max_depth:
            continue
        for key, ckey, action, data in cands:
            nk = model.next_key(sk, key)
            if nk is not None:
                if nk not in seen:
                    seen.add(nk); q.append((nk, path + [(key, action, data)]))
                continue
            pred, src = model.predict(o_list, g, ckey)
            if pred is None:
                continue
            nk = state_key(pred)
            if nk == sk or nk in seen:
                continue
            # render the predicted grid for the moved objects
            g2 = g
            for a, b in zip(o_list, pred):
                if (a.y0, a.x0) != (b.y0, b.x0):
                    g2 = render_move(g2, a, b, bg)
            objs_cache.setdefault(nk, (pred, g2))
            seen.add(nk); q.append((nk, path + [(key, action, data)]))
    return best if best else (None, None)

# ---- agents/base.py ----



ACTION_BY_ID = {a.value: a for a in GameAction}


class BaseAgent:
    """Minimal stand-in for agents.agent.Agent with the same two abstract methods."""

    MAX_ACTIONS: int = 2000

    def __init__(self, game_id: str, seed: int = 0, **kwargs: Any) -> None:
        self.game_id = game_id
        self.rng = random.Random(seed)

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        return latest_frame.state is GameState.WIN

    def choose_action(self, frames: list[FrameData], latest_frame: FrameData) -> GameAction:
        raise NotImplementedError

    # helpers shared by agents -------------------------------------------------------------
    @staticmethod
    def legal_actions(latest_frame: FrameData) -> list[GameAction]:
        ids = latest_frame.available_actions or [1, 2, 3, 4, 5, 6, 7]
        return [ACTION_BY_ID[i] for i in ids if i in ACTION_BY_ID and i != 0]

    def random_action(self, latest_frame: FrameData) -> GameAction:
        a = self.rng.choice(self.legal_actions(latest_frame))
        if a.is_complex():
            a.set_data({"x": self.rng.randint(0, 63), "y": self.rng.randint(0, 63)})
        return a

# ---- agents/model_based.py ----
"""H1 agent: learn an object-level effect model online, plan in it toward information gain, keep the
model across levels (memory=True) or reset it at each level (memory=False)."""




class ModelBasedAgent(BaseAgent):
    def __init__(self, game_id: str, seed: int = 0, memory: bool = True, max_depth: int = 12, max_nodes: int = 1500,
                 click_lattice: int = 8, **kwargs):
        super().__init__(game_id, seed)
        self.memory = memory
        self.max_depth, self.max_nodes, self.click_lattice = max_depth, max_nodes, click_lattice
        self.model = EffectModel()
        self.tried: dict[int, set] = {}
        self.objs_cache: dict[int, tuple[list[Obj], np.ndarray]] = {}
        self.plan: list = []
        self.predicted_next: int | None = None
        self.prev: tuple[list[Obj], np.ndarray, object, object] | None = None  # objs, grid, key, ckey
        self.levels_seen = 0
        self.n_plans = 0; self.n_plan_steps = 0; self.n_replans_on_error = 0; self.n_fallback = 0
        self.pred_by_source = {"memo": [0, 0], "model": [0, 0]}
        self.row_changes = np.zeros(64, dtype=np.int32); self.row_steps = 0; self.prev_grid: np.ndarray | None = None
        self.volatile_rows: np.ndarray = np.zeros(64, dtype=bool)
        self.terrain_colours: set[int] = set()

    def _update_volatility(self, grid: np.ndarray, after_reset: bool) -> None:
        if self.prev_grid is not None and not after_reset:
            self.row_changes += (grid != self.prev_grid).any(axis=1); self.row_steps += 1
            if self.row_steps >= 10:
                self.volatile_rows = (self.row_changes / self.row_steps) > 0.8
        self.prev_grid = grid

    TERRAIN_FRAC = 0.20  # a colour covering >= 20% of the frame is terrain (background, floor): kept in the grid for
                         # blocking checks but its objects are excluded from the state key

    def perceive(self, grid: np.ndarray) -> list[Obj]:
        objs = segment(grid, drop_background=False)
        counts = np.bincount(grid.ravel(), minlength=16)
        terrain = {int(c) for c in np.where(counts >= self.TERRAIN_FRAC * grid.size)[0]}
        if not terrain:
            terrain = {int(counts.argmax())}
        self.terrain_colours = terrain
        if self.volatile_rows.any():
            objs = [o for o in objs if not self.volatile_rows[o.y0:o.y1].all()]
        return [o for o in objs if o.colour not in terrain]

    # ---- candidate actions ---------------------------------------------------------------------
    def _keys_for(self, latest: FrameData):
        legal = self.legal_actions(latest)

        def keys(objs: list[Obj]):
            out = []
            for a in legal:
                if a.is_complex():
                    seen_cls = set()
                    for o in objs:
                        cy, cx = o.centre
                        cls = ("click",) + ident(o)
                        out.append((("click",) + ident(o) + (o.y0, o.x0), cls, a, {"x": int(cx), "y": int(cy)}))
                        seen_cls.add(cls)
                    step = 64 // self.click_lattice
                    for y in range(step // 2, 64, step):
                        for x in range(step // 2, 64, step):
                            out.append((("click", "lattice", y, x), ("click", "lattice"), a, {"x": x, "y": y}))
                else:
                    out.append((a.value, a.value, a, {}))
            return out
        return keys

    # ---- main -------------------------------------------------------------------------------------
    def choose_action(self, frames: list[FrameData], latest: FrameData) -> GameAction:
        if latest.state in (GameState.NOT_PLAYED, GameState.GAME_OVER) or not latest.frame:
            self.prev = None; self.plan = []; self.predicted_next = None
            return GameAction.RESET
        grid = np.asarray(latest.frame[-1], dtype=np.int16)
        self._update_volatility(grid, after_reset=self.prev is None)
        objs = self.perceive(grid)
        sk = state_key(objs)
        self.objs_cache[sk] = (objs, grid)

        # level transition bookkeeping
        if latest.levels_completed > self.levels_seen:
            self.levels_seen = latest.levels_completed
            self.tried = {}; self.plan = []; self.predicted_next = None
            if not self.memory:
                self.model = EffectModel(); self.objs_cache = {sk: (objs, grid)}
            self.prev = None

        # learn from the last transition
        if self.prev is not None:
            p_objs, p_grid, p_key, p_ckey = self.prev
            self.model.observe(p_objs, p_grid, p_ckey, objs)
            self.model.transitions[(state_key(p_objs), p_key)] = sk
            if self.predicted_next is not None:
                src = self._pred_src
                self.pred_by_source[src][0] += 1
                self.pred_by_source[src][1] += int(self.predicted_next == sk)
                self.model.record_prediction(self.predicted_next, sk, p_ckey if src == "model" else None)
                if self.predicted_next != sk:
                    self.plan = []; self.n_replans_on_error += 1
            self.predicted_next = None

        keys_for = self._keys_for(latest)
        if not self.plan:
            path, final = plan_to_information(self.model, objs, grid, keys_for, self.tried, self.objs_cache,
                                              self.max_depth, self.max_nodes)
            if path is not None:
                self.plan = list(path) + [final[:1] + (final[2], final[3])] if final else list(path)
                # normalise entries to (key, action, data)
                self.plan = [(e[0], e[-2], e[-1]) for e in self.plan]
                self.n_plans += 1
        if self.plan:
            key, action, data = self.plan.pop(0)
            self.n_plan_steps += 1
        else:
            self.n_fallback += 1
            cands = keys_for(objs)
            untried = [c for c in cands if c[0] not in self.tried.get(sk, set())]
            pool = untried or cands
            key, ckey, action, data = self.rng.choice(pool)
        ckey = key if not isinstance(key, tuple) else (("click",) + tuple(key[1:4]) if key[1] != "lattice" else ("click", "lattice"))
        self.tried.setdefault(sk, set()).add(key)
        # predict for accuracy accounting
        nk = self.model.next_key(sk, key)
        if nk is not None:
            self.predicted_next, self._pred_src = nk, "memo"
        else:
            pred, src = self.model.predict(objs, grid, ckey)
            if pred is not None:
                self.predicted_next, self._pred_src = state_key(pred), "model"
        self.prev = (objs, grid, key, ckey)
        if action.is_complex():
            action.set_data(data)
        return action

    def stats(self) -> dict:
        return {"memory": self.memory, "plans": self.n_plans, "plan_steps": self.n_plan_steps,
                "replans_on_error": self.n_replans_on_error, "fallback_steps": self.n_fallback,
                "pred_total": self.model.n_pred, "pred_acc": None if self.model.accuracy is None else round(self.model.accuracy, 3),
                "pred_by_source": {k: (v[0], round(v[1] / v[0], 3) if v[0] else None) for k, v in self.pred_by_source.items()},
                "movers": {str(k): [sorted(map(str, g[0])), list(g[1])] for k in self.model.moves if (g := self.model.modal_group(k))},
                "volatile_rows": [int(i) for i in np.where(self.volatile_rows)[0]], "terrain_colours": sorted(self.terrain_colours),
                "known_noops": [str(k) for k in self.model.effects if self.model.is_known_noop(k)],
                "key_reliability": {str(k): [v[0], round(v[1] / v[0], 2)] for k, v in self.model.key_pred.items() if v[0]}}

# ======================================================================================
# Kaggle framework adapter (ARC-AGI-3-Agents contract)
# ======================================================================================
import os as _os, time as _time
from agents.agent import Agent as _FrameworkAgent

_START = _time.time()
_BUDGET_S = float(_os.getenv("ARC3_BUDGET_S", "30600"))
_MAX_ACTIONS = int(_os.getenv("ARC3_MAX_ACTIONS", "1500"))
_MEMORY = _os.getenv("ARC3_MEMORY", "1") == "1"


class MyAgent(_FrameworkAgent):
    """ModelBasedAgent behind the framework's Agent interface. One instance per game."""

    MAX_ACTIONS = _MAX_ACTIONS

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.impl = ModelBasedAgent(game_id=self.game_id, seed=0, memory=_MEMORY)
        self._n = 0

    @property
    def name(self) -> str:
        return f"{super().name}.modelbased.mem{int(_MEMORY)}"

    def is_done(self, frames, latest_frame) -> bool:
        if latest_frame.state is GameState.WIN:
            return True
        if _time.time() - _START > _BUDGET_S:
            return True
        return self._n >= _MAX_ACTIONS

    def choose_action(self, frames, latest_frame) -> GameAction:
        self._n += 1
        try:
            return self.impl.choose_action(frames, latest_frame)
        except Exception as e:  # never crash the swarm thread
            if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
                return GameAction.RESET
            acts = [a for a in GameAction if a is not GameAction.RESET and a.value in (latest_frame.available_actions or [1, 2, 3, 4])]
            a = acts[self._n % len(acts)] if acts else GameAction.ACTION1
            if a.is_complex():
                a.set_data({"x": 32, "y": 32})
            return a


In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Wait for the gateway to be ready
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 --retry-max-time 600 http://gateway:8001/api/games

    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents /kaggle/working/ARC-AGI-3-Agents
    !cp /kaggle/working/my_agent.py /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type, cast
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    "random": Random,
    "myagent": MyAgent,
}
""")

    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
""")

    !cd /kaggle/working/ARC-AGI-3-Agents && MPLBACKEND=agg python main.py --agent myagent


In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Non-competition mode: produce a dummy submission file so "Save & Run All" succeeds
    import pandas as pd
    submission = pd.DataFrame(data=[['1_0', '1', True, 1]], columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    submission.head()
